# Below defined class of code is to build the SAR FAISS database - sar_gallery_192bit.index

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import your classes here (BEN14KDataset, Antariksh_ViT_Encoders)
# Make sure they are defined in the same script or imported

def build_faiss_gallery():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Building FAISS Index on: {device}")

    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    
    val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    # Load Model (Fixed PyTorch Warning)
    encoders = Antariksh_ViT_Encoders().to(device)
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    encoders.eval()

    # FAISS setup
    hash_bits = 192
    index = faiss.IndexBinaryFlat(hash_bits)
    
    gallery_patch_ids = val_dataset.patch_ids
    gallery_labels = val_dataset.labels
    
    print("\nExtracting SAR Features & Binarizing...")
    with torch.no_grad():
        for batch_idx, (s1_imgs, s2_imgs) in enumerate(tqdm(val_loader)):
            s1_imgs = s1_imgs.to(device)
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            s1_features = s1_latent.mean(dim=1).cpu().numpy()
            
            binary_hashes = (s1_features > 0).astype(bool)
            packed_hashes = np.packbits(binary_hashes, axis=1)
            index.add(packed_hashes)

    faiss.write_index_binary(index, "sar_gallery_192bit.index")
    
    metadata_df = pd.DataFrame({'patch_id': gallery_patch_ids, 'labels': gallery_labels})
    metadata_df.to_pickle("gallery_metadata.pkl")
    
    print(f"\n✅ Successfully built and saved FAISS database with {index.ntotal} SAR images!")

build_faiss_gallery()

# Below defined class of code is to build the optical FAISS database - optical_gallery_192bit.index

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import your classes
# import BEN14KDataset, Antariksh_ViT_Encoders

def build_optical_gallery():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Building Optical FAISS Index on: {device}")

    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    
    val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    encoders = Antariksh_ViT_Encoders().to(device)
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    encoders.eval()

    hash_bits = 192
    index = faiss.IndexBinaryFlat(hash_bits)
    
    print("\nExtracting Optical (S2) Features & Binarizing...")
    with torch.no_grad():
        for batch_idx, (s1_imgs, s2_imgs) in enumerate(tqdm(val_loader)):
            s2_imgs = s2_imgs.to(device)
            
            # Optical Encoder se features nikal rahe hain
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            s2_features = s2_latent.mean(dim=1).cpu().numpy()
            
            binary_hashes = (s2_features > 0).astype(bool)
            packed_hashes = np.packbits(binary_hashes, axis=1)
            index.add(packed_hashes)

    faiss.write_index_binary(index, "optical_gallery_192bit.index")
    print(f"\n✅ Successfully built Optical FAISS database with {index.ntotal} images!")

if __name__ == "__main__":
    build_optical_gallery()

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# Apni classes (BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor) yahan ensure karein
# from app import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def build_optical_gallery_with_predictor():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Building Advanced Optical FAISS Index on: {device}")

    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    
    val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    # 1. Load BOTH Encoder and Predictor
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    encoders.eval()
    predictor.eval()

    hash_bits = 192
    index = faiss.IndexBinaryFlat(hash_bits)
    
    print("\nExtracting Optical Features & Translating to Common Space...")
    with torch.no_grad():
        for batch_idx, (s1_imgs, s2_imgs) in enumerate(tqdm(val_loader)):
            s2_imgs = s2_imgs.to(device)
            
            # --- THE FIX: Use Predictor to translate Optical to Common Space ---
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            
            # Ab hum Predictor ke output ko database mein save karenge
            s2_features = predicted_sar.mean(dim=1).cpu().numpy()
            
            binary_hashes = (s2_features > 0).astype(bool)
            packed_hashes = np.packbits(binary_hashes, axis=1)
            index.add(packed_hashes)

    faiss.write_index_binary(index, "optical_gallery_192bit.index")
    print(f"\n✅ Successfully built Advanced Optical FAISS database with {index.ntotal} images!")

if __name__ == "__main__":
    build_optical_gallery_with_predictor()